In [ ]:
# Part 0 Part 0: Environment & Dataset Acquisition

In [ ]:
# Step 1

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
print("Environment successfully configured!")

Environment successfully configured!


In [2]:
# Step 2: Generate Synthetic Customer Transaction Logs

In [4]:
np.random.seed(42)
n_transactions = 15000
start_date = dt.datetime(2025, 1, 1)

customer_ids = [f"CUST_{np.random.randint(1000, 2000)}" for _ in range(n_transactions)]
transaction_ids = [f"TXN_{str(i).zfill(6)}" for i in range(1, n_transactions + 1)]
random_days = np.random.randint(0, 365, size=n_transactions)
transaction_dates = [start_date + dt.timedelta(days=int(d)) for d in random_days]
order_values = np.round(np.random.exponential(scale=75, size=n_transactions) + 10, 2)

raw_df = pd.DataFrame({
'TransactionID': transaction_ids,
'CustomerID': customer_ids,
'TransactionDate': transaction_dates,
'OrderValue': order_values
})

raw_df.to_csv('raw_customer_transactions.csv', index=False)
print(f"Dataset successfully created with {len(raw_df)} transaction logs across {raw_df['CustomerID'].nunique()} unique customers!")
raw_df.head()

Dataset successfully created with 15000 transaction logs across 1000 unique customers!


,TransactionID,CustomerID,TransactionDate,OrderValue
0,TXN_000001,CUST_1102,2025-05-11,53.44
1,TXN_000002,CUST_1435,2025-03-08,318.28
2,TXN_000003,CUST_1860,2025-08-16,73.14
3,TXN_000004,CUST_1270,2025-06-12,27.00
4,TXN_000005,CUST_1106,2025-01-15,35.98


In [5]:
# Part 1: Feature Engineering  Computing RFM Metrics

In [6]:
# Step 1: Aggregate Customer-Level Metrics

In [7]:
snapshot_date = raw_df['TransactionDate'].max() + dt.timedelta(days=1)

rfm_df = raw_df.groupby('CustomerID').agg({
'TransactionDate': lambda x: (snapshot_date - x.max()).days,
'TransactionID': 'count',
'OrderValue': 'sum'
}).reset_index()

rfm_df.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
print(f"RFM aggregated for {len(rfm_df)} customers.")
rfm_df.head()

RFM aggregated for 1000 customers.


,CustomerID,Recency,Frequency,Monetary
0,CUST_1000,15,19,1586.42
1,CUST_1001,39,15,1061.24
2,CUST_1002,10,12,1240.64
3,CUST_1003,40,10,1003.76
4,CUST_1004,30,10,1058.63


In [8]:
# Part 2: Feature Scaling & K-Means Clustering

In [9]:
# Step 1: Standardize RFM Features

In [10]:
rfm_features = rfm_df[['Recency', 'Frequency', 'Monetary']]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency_Scaled', 'Frequency_Scaled', 'Monetary_Scaled'])
rfm_scaled_df.head()

,Recency_Scaled,Frequency_Scaled,Monetary_Scaled
0,-0.404689,1.004151,0.700345
1,0.604721,0.000000,-0.504873
2,-0.614983,-0.753113,-0.093174
3,0.646779,-1.255188,-0.636782
4,0.226192,-1.255188,-0.510862


In [11]:
# Step 2: Train K-Means Model & Discover Customer Segments

In [12]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_df['Cluster'] = kmeans.fit_predict(rfm_scaled)


cluster_summary = rfm_df.groupby('Cluster').agg({
'Recency': 'mean',
'Frequency': 'mean',
'Monetary': ['mean', 'count']
}).round(2)
print("Cluster Summary (Mean Values):")
print(cluster_summary)

Cluster Summary (Mean Values):
        Recency Frequency Monetary      
           mean      mean     mean count
Cluster                                 
0         17.77     16.01  1356.02   430
1         20.62     11.42   885.01   295
2         78.38     11.85  1015.82   104
3         16.06     20.56  1938.18   171


In [13]:
# Step 3: Map Clusters to Actionable Customer Personas

In [14]:
def assign_persona(row):
  if row['Cluster'] == 0:
    return 'At-Risk / Lost'
  elif row['Cluster'] == 1:
    return 'Champions'

  elif row['Cluster'] == 2:
    return 'Loyal Customers'
  else:
    return 'Recent / New Visitors'
rfm_df['Customer_Persona'] = rfm_df.apply(assign_persona, axis=1)

In [15]:
rfm_df.to_csv('rfm_segmented_customers.csv', index=False)
print("Segmented dataset exported successfully as 'rfm_segmented_customers.csv'!")
rfm_df.head()

Segmented dataset exported successfully as 'rfm_segmented_customers.csv'!


,CustomerID,Recency,Frequency,Monetary,Cluster,Customer_Persona
0,CUST_1000,15,19,1586.42,3,Recent / New Visitors
1,CUST_1001,39,15,1061.24,0,At-Risk / Lost
2,CUST_1002,10,12,1240.64,1,Champions
3,CUST_1003,40,10,1003.76,1,Champions
4,CUST_1004,30,10,1058.63,1,Champions


In [16]:
# Part 3: Tableau Dashboard Construction